# NB03: Phylogenetic Distribution of AMR Genes

**Goal**: Test H2 — is AMR gene density phylogenetically structured? Identify AMR hotspot lineages. Analyze how AMR density correlates with pangenome openness and genome count across the tree.

**Depends on**: NB01 outputs (`data/amr_census.csv`, `data/amr_species_summary.csv`)

**Outputs**:
- `../data/amr_by_taxonomy.csv` — AMR stats at each taxonomic rank
- `../figures/amr_phylogenetic_*.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')

plt.rcParams.update({
    'figure.figsize': (12, 6), 'figure.dpi': 150, 'font.size': 11,
    'axes.titlesize': 13, 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})

amr = pd.read_csv(DATA_DIR / 'amr_census.csv')
species = pd.read_csv(DATA_DIR / 'amr_species_summary.csv')
print(f"Loaded {len(amr):,} AMR clusters, {len(species):,} species with AMR")

## 1. Taxonomic Hierarchy: AMR at Every Rank

Aggregate AMR counts at phylum, class, order, family, and genus levels.

In [ ]:
# AMR stats at each taxonomic rank
ranks = ['phylum', 'class', 'order', 'family', 'genus']
rank_summaries = {}

for rank in ranks:
    summary = species.groupby(rank).agg(
        n_species=('gtdb_species_clade_id', 'count'),
        total_amr=('n_amr', 'sum'),
        mean_amr=('n_amr', 'mean'),
        median_amr=('n_amr', 'median'),
        max_amr=('n_amr', 'max'),
        mean_openness=('openness', 'mean'),
        mean_genomes=('no_genomes', 'mean')
    ).round(2)
    summary = summary.sort_values('total_amr', ascending=False)
    rank_summaries[rank] = summary
    
    print(f"\n=== {rank.upper()} level: Top 15 by total AMR clusters ===")
    print(summary.head(15).to_string())

# Save genus-level (most granular useful rank)
rank_summaries['genus'].to_csv(DATA_DIR / 'amr_by_genus.csv')
rank_summaries['family'].to_csv(DATA_DIR / 'amr_by_family.csv')
print(f"\nSaved genus and family summaries to data/")

## 2. AMR Hotspot Families

Which families have disproportionately high AMR density (AMR clusters per species)?

In [ ]:
# Family-level hotspot analysis
fam = rank_summaries['family'].copy()
fam_enough = fam[fam['n_species'] >= 5].copy()  # at least 5 species
overall_mean = species['n_amr'].mean()

print(f"Overall mean AMR clusters per species: {overall_mean:.1f}")
print(f"Families with >= 5 species: {len(fam_enough):,}")

# Top families by mean AMR density
print(f"\n=== Top 20 Families by Mean AMR Clusters/Species (min 5 species) ===")
top_fam = fam_enough.nlargest(20, 'mean_amr')
print(top_fam[['n_species', 'total_amr', 'mean_amr', 'median_amr', 'mean_openness']].to_string())

# Visualization: Top 20 families
fig, ax = plt.subplots(figsize=(14, 8))
top_fam_sorted = top_fam.sort_values('mean_amr')
colors = plt.cm.Reds(top_fam_sorted['mean_amr'] / top_fam_sorted['mean_amr'].max())

ax.barh(range(len(top_fam_sorted)), top_fam_sorted['mean_amr'], color=colors)
ax.set_yticks(range(len(top_fam_sorted)))
ax.set_yticklabels([idx.replace('f__', '') for idx in top_fam_sorted.index], fontsize=9)
ax.axvline(overall_mean, color='gray', linestyle='--', alpha=0.7, label=f'Overall mean ({overall_mean:.1f})')
ax.set_xlabel('Mean AMR Clusters per Species')
ax.set_title('Top 20 AMR Hotspot Families (min 5 species)')
ax.legend()

for i, (_, row) in enumerate(top_fam_sorted.iterrows()):
    ax.text(row['mean_amr'] + 0.2, i, f'n={int(row["n_species"])}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_hotspot_families.png')
plt.show()

## 3. AMR Mechanism Distribution Across Phyla

Do different phyla rely on different resistance mechanisms?

In [ ]:
# Mechanism classification (re-derive if needed)
if 'mechanism' not in amr.columns:
    def classify_mechanism(product):
        if pd.isna(product):
            return 'Unknown'
        p = product.lower()
        if any(k in p for k in ['efflux', 'pump', 'transporter', 'exporter', 'permease']):
            return 'Efflux'
        if any(k in p for k in ['beta-lactamase', 'lactamase', 'carbapenemase']):
            return 'Beta-lactamase'
        if any(k in p for k in ['acetyltransferase', 'phosphotransferase', 'nucleotidyltransferase',
                                 'aminoglycoside-modifying', 'inactivat', 'hydrolase']):
            return 'Enzymatic inactivation'
        if any(k in p for k in ['methyltransferase', 'ribosomal protection', 'target protect']):
            return 'Target modification'
        if any(k in p for k in ['penicillin-binding', 'pbp', 'vancomycin', 'lipid a', 'mcr-']):
            return 'Cell wall modification'
        return 'Other'
    amr['mechanism'] = amr['amr_product'].apply(classify_mechanism)

# Cross-tabulate: phylum x mechanism
top_phyla = amr['phylum'].value_counts().head(8).index
top_mechs = amr['mechanism'].value_counts().head(6).index

ct = pd.crosstab(amr[amr['phylum'].isin(top_phyla)]['phylum'],
                 amr[amr['mechanism'].isin(top_mechs)]['mechanism'],
                 normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 7))
ct.plot(kind='barh', stacked=True, ax=ax, colormap='Set2')
ax.set_xlabel('% of AMR Genes')
ax.set_title('AMR Mechanism Composition by Phylum')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_yticklabels([t.replace('p__', '') for t in ct.index], fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_mechanism_by_phylum.png')
plt.show()

# Chi-squared test for mechanism independence from phylum
ct_counts = pd.crosstab(amr[amr['phylum'].isin(top_phyla)]['phylum'],
                         amr[amr['mechanism'].isin(top_mechs)]['mechanism'])
chi2, p, dof, expected = stats.chi2_contingency(ct_counts)
print(f"\nChi-squared test (mechanism x phylum independence):")
print(f"  chi2={chi2:.1f}, dof={dof}, p={p:.2e}")
print(f"  Interpretation: {'Mechanisms differ by phylum' if p < 0.05 else 'No significant difference'}")

## 4. AMR Density vs Pangenome Openness — Controlling for Phylogeny

The NB01 correlation between AMR count and openness could be driven by phylogeny. Test within-phylum correlations.

In [ ]:
# Within-phylum correlations: AMR count vs openness
from scipy.stats import spearmanr

top_phyla_list = species['phylum'].value_counts().head(10).index.tolist()

print("=== Within-Phylum: AMR Count vs Pangenome Openness ===\n")
print(f"{'Phylum':<30} {'N spp':>6} {'rho':>8} {'p-value':>12} {'Sig':>5}")
print("-" * 65)

results = []
for phy in top_phyla_list:
    sub = species[species['phylum'] == phy].dropna(subset=['openness', 'n_amr'])
    if len(sub) >= 10:
        rho, p = spearmanr(sub['openness'], sub['n_amr'])
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"{phy.replace('p__',''):<30} {len(sub):>6} {rho:>8.3f} {p:>12.2e} {sig:>5}")
        results.append({'phylum': phy, 'n': len(sub), 'rho': rho, 'p': p})

# Overall
rho_all, p_all = spearmanr(species['openness'].dropna(), species['n_amr'].dropna())
print(f"\n{'OVERALL':<30} {len(species):>6} {rho_all:>8.3f} {p_all:>12.2e}")

# How many phyla show positive vs negative correlation?
pos = sum(1 for r in results if r['rho'] > 0)
neg = sum(1 for r in results if r['rho'] < 0)
print(f"\nPhyla with positive correlation: {pos}/{len(results)}")
print(f"Phyla with negative correlation: {neg}/{len(results)}")

In [ ]:
# Faceted scatter: AMR vs openness by phylum (top 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flatten()

for i, phy in enumerate(top_phyla_list[:6]):
    ax = axes_flat[i]
    sub = species[species['phylum'] == phy].dropna(subset=['openness', 'n_amr'])
    ax.scatter(sub['openness'], sub['n_amr'], alpha=0.4, s=12, color='#e74c3c')
    
    rho, p = spearmanr(sub['openness'], sub['n_amr'])
    ax.set_title(f"{phy.replace('p__', '')} (n={len(sub)}, rho={rho:.2f})", fontsize=10)
    ax.set_xlabel('Openness')
    ax.set_ylabel('AMR Clusters')
    
    # Trend line
    if len(sub) > 10:
        z = np.polyfit(sub['openness'], sub['n_amr'], 1)
        x_line = np.linspace(sub['openness'].min(), sub['openness'].max(), 100)
        ax.plot(x_line, np.polyval(z, x_line), 'k--', alpha=0.5)

plt.suptitle('AMR Count vs Pangenome Openness by Phylum', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_openness_by_phylum.png')
plt.show()